# Tuning against race time

**You have about 30 knobs and 0.3 s of noise. Which changes are real?**

`agents/unit-11.py` is the agent the book has built so far: Unit 9's racing line, Unit 8's measured grip, Unit 7's speed profile, Unit 10's MPC and Unit 11's fallback.
Every choice in it is a **module constant**, a line like `A_DEC = 5.0` at the top of the file, and every one was set by hand, one unit at a time.
The setup cell below counts them, straight from the file.

Race time is a number a computer can push down: change some constants, race, keep the change if the race got faster, repeat.
The probe record of plan `roar-012` did exactly that.
A **coordinate search** over six of the constants (`MU_FRACTION`, `SPEED_KP`, `A_DEC`, `A_ACC`, `SPEED_KI`, `TARGET_SPEED`) raced 200 three-lap races on the five training tracks, kept every step that was faster by more than the noise and clean, and finished **48.9 s faster** over the 15 laps.
Then the same setting raced a track the search never saw, `test-circuit-seed2026` (Checkpoint A's track: in this unit, the **validation** track):

In [ ]:
import cProfile
import math
import pstats
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
import racekit
from racekit import write_variant
from racekit.cars import carla_like
from racekit.measure import noise_aware_better
from racekit.runner import load_submission, run_race, shim_installed
from racekit.tracks import load_track
from scipy.optimize import lsq_linear

AGENTS = Path("../../projects/racekit/agents")
UNIT_11, UNIT_12 = AGENTS / "unit-11.py", AGENTS / "unit-12.py"
workdir = Path(mkdtemp(prefix="roar-unit-12-"))
TICK = 0.05 + 1e-9  # s: race times are pinned to one control tick
DT = 0.05  # s: one control tick
CLEAN_ERROR = 2.0  # m: the clean rule's limit on the error from the race's own line
NO_TIMING = {"MPC_BUDGET_MS": None}  # every race here: no wall clock inside the race (Unit 10, section 7)
TRAINING = ("test-circuit", "test-circuit-seed1", "test-circuit-seed2", "test-circuit-seed3", "test-circuit-seed4")
VALIDATION = "test-circuit-seed2026"  # never in an objective in this unit; it judges a setting already chosen
# Unit 8's grip probes: the measured mu * g of each grip section (m/s^2, rounded to 0.01), per track.
# The agents ship test-circuit's; another track's values are written in with write_variant (Units 9-11).
MEASURED_MU_G = {
    "test-circuit-seed1": (7.72, 9.80, 9.36, 8.38, 9.53, 7.56),
    "test-circuit-seed2": (9.06, 8.37, 8.49, 8.86, 9.78, 9.92),
    "test-circuit-seed3": (9.55, 8.89, 8.15, 9.94, 8.50, 9.09),
    "test-circuit-seed4": (8.79, 8.53, 9.10, 8.54, 8.41, 9.97),
    "test-circuit-seed2026": (7.70, 7.69, 7.70, 9.81, 8.16, 8.04),
}


# Unit 9's line solver: a cubic B-spline of offsets, the smoothest line by bounded least squares.
def spline_basis(n, step=10):
    m = n // step
    basis = np.zeros((n, m))
    for i in range(n):
        t = i * m / n
        j = math.floor(t)
        u = t - j
        weights = ((1 - u) ** 3 / 6, (3 * u ** 3 - 6 * u ** 2 + 4) / 6, (-3 * u ** 3 + 3 * u ** 2 + 3 * u + 1) / 6,
                   u ** 3 / 6)
        for k in range(4):
            basis[i, (j - 1 + k) % m] += weights[k]
    return basis


def normals(x, y):
    dx, dy = np.roll(x, -1) - np.roll(x, 1), np.roll(y, -1) - np.roll(y, 1)
    h = np.hypot(dx, dy)
    return -dy / h, dx / h


def second_difference(values):
    return np.roll(values, -1, axis=0) - 2.0 * values + np.roll(values, 1, axis=0)


def solve_line(track, margin=2.5, pin=30):
    x, y = track.x, track.y
    nx, ny = normals(x, y)
    basis = spline_basis(len(x))
    A = np.vstack([second_difference(nx[:, None] * basis), second_difference(ny[:, None] * basis)])
    b = -np.concatenate([second_difference(x), second_difference(y)])
    rows, columns = basis.shape
    index = np.arange(rows)
    pinned = (index < pin) | (index >= rows - pin)
    high = np.zeros(columns)
    for j in range(columns):
        support = basis[:, j] > 0
        high[j] = 0.0 if np.sum(pinned[support]) > 0 else np.min(track.half_width[support] - margin)
    Q, R = np.linalg.qr(A)
    return lsq_linear(R, Q.T @ b, bounds=(-high - 1e-12, high + 1e-12), method="bvls").x


# Every track's line: test-circuit's as the agents ship it, every other track's solved here (Unit 9).
with shim_installed():
    Unit11, _ = load_submission(UNIT_11)
control, line_offsets = {}, {}
for name in TRAINING + (VALIDATION,):
    track = load_track(name)
    if name == "test-circuit":
        line_offsets[name] = np.array(Unit11(track.waypoints, None).offsets)
    else:
        control[name] = solve_line(track)
        line_offsets[name] = spline_basis(len(track)) @ control[name]


written = []  # every variant file written so far


def variant(source, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, each written as its repr.
    path = workdir / f"v{len(written):03d}.py"
    written.append(path)
    return write_variant(source, path, {key: repr(value) for key, value in constants.items()})


def on_track(track_name, **constants):
    # The constants a race on this track needs: its own line and its measured grip (test-circuit: as shipped).
    if track_name != "test-circuit":
        constants = {"LINE_CONTROL": tuple(float(value) for value in control[track_name]),
                     "MU_G": MEASURED_MU_G[track_name], **constants}
    return constants


def race(setting, track="test-circuit", seed=0, laps=1, source=UNIT_11, k=None, max_seconds=450.0):
    # One race of `source` with the constants in `setting` changed and the timing off, on `track` with its line
    # and grip, on the carla-like car with its noise (car k of Unit 11's perturbed family if k is given).
    path = variant(source, **on_track(track, **setting), **NO_TIMING)
    return run_race(path, track, car=carla_like(noise=True, perturb=k), seed=seed, laps=laps, max_seconds=max_seconds)


def line_error(result):
    # Unit 9: the largest |cross-track error from the race's own line| over the driving rows (held == 0).
    offsets = line_offsets[result.track]
    telemetry = result.telemetry
    driving = telemetry["held"] == 0
    target = offsets[(telemetry["progress"][driving] + 1) % len(offsets)]
    return float(np.max(np.abs(telemetry["offset"][driving] - target)))


def clean(result):
    return result.finished and result.collisions == 0 and line_error(result) <= CLEAN_ERROR


def same_race(a, b):
    # Two races are the same race when every telemetry row is equal, not only the time.
    return len(a.telemetry) == len(b.telemetry) and all(
        np.sum(a.telemetry[name] != b.telemetry[name]) == 0 for name in a.telemetry.dtype.names)


NAME_CHARACTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_"


def module_constants(path):
    # name -> the text after " = ", for every line of the file that starts with an UPPER_CASE name and " = ".
    constants = {}
    with open(path) as handle:
        for line in handle:
            cuts = [i for i in range(len(line) - 2) if line[i:i + 3] == " = "]
            if cuts and line[0] in NAME_CHARACTERS[:26] and all(c in NAME_CHARACTERS for c in line[:cuts[0]]):
                constants[line[:cuts[0]]] = line[cuts[0] + 3:-1]
    return constants


knobs_in_file = module_constants(UNIT_11)
print(f"agents/unit-11.py has {len(knobs_in_file)} module constants, from {list(knobs_in_file)[0]} "
      f"to {list(knobs_in_file)[-1]}")
assert len(knobs_in_file) == 48

# The probe record of plan roar-012 (p3, U12.4): a coordinate search on the training family. The objective: for
# each race seed (0 and 1), the sum of the 3-lap race times on the five training tracks; any race not clean
# rejects a setting. (races so far, the incumbent's objective in s) each time the search took a step:
P3_COORDINATE = [(10, 1383.15), (30, 1370.93), (40, 1359.33), (50, 1348.15), (100, 1341.85), (130, 1334.25), (200, 1334.25)]
SPEED_SETTING = {"MU_FRACTION": 0.60, "A_DEC": 5.5, "A_ACC": 4.5}  # its result, on the grid (from 0.54, 5.0, 4.0)
# The same two agents on the validation track, 3 laps, race seeds 0-4 (premise U12.4 and U12.1):
SEED2026_UNIT_11 = [282.05, 282.00, 282.00, 282.00, 281.90]  # agents/unit-11.py: 5/5 clean
SEED2026_SPEED = [318.25, 316.45, 318.85, 317.00, 318.85]  # the speed setting: 0/5 clean, up to 5.25 m off its line
print(f"training family (10 races): {P3_COORDINATE[0][1]:.2f} -> {P3_COORDINATE[-1][1]:.2f} s, "
      f"{P3_COORDINATE[0][1] - P3_COORDINATE[-1][1]:.2f} s faster over 15 laps, every race clean")
print(f"{VALIDATION}: unit-11.py median {np.median(SEED2026_UNIT_11):.2f} s (5/5 clean); "
      f"the speed setting median {np.median(SEED2026_SPEED):.2f} s (0/5 clean)")

fig, (ax_search, ax_valid) = plt.subplots(1, 2, figsize=(11, 3.5))
races, objective = zip(*P3_COORDINATE)
ax_search.step(races, objective, where="post")
ax_search.set_xlabel("races")
ax_search.set_ylabel("objective (s, 15 laps)")
ax_search.set_title("the speed search on the training tracks")
ax_valid.plot(range(5), SEED2026_UNIT_11, "o", label="unit-11.py (clean)")
ax_valid.plot(range(5), SEED2026_SPEED, "x", markersize=9, label="the speed setting (not clean)")
ax_valid.set_xlabel("race seed")
ax_valid.set_ylabel("race time (s, 3 laps)")
ax_valid.set_title(f"the validation track, {VALIDATION}")
ax_valid.legend()
fig.tight_layout()
plt.show()

48 constants.
About 30 of them can change a race (§2 counts them exactly); the rest switch modes or are never read.

On the training tracks the search's setting is clean in all 10 of its races and 48.9 s faster: more than 3 s a lap.
On the validation track it is **not clean in any of five races**, it runs up to 5.25 m off its line, and it is 36 s slower than the hand-tuned `unit-11.py`.
On Unit 11's perturbed cars it is clean on only 7 of 24.
The search did exactly what it was asked: be fast on these five tracks, on this one car.
It was never asked to be fast anywhere else.

This unit tunes against race time, and learns where that goes wrong.
§1 defines the objective and measures its noise; §2 finds which constants do anything; §3 runs three searches (coordinate, random and an evolution strategy) and tastes a fourth; §4 shows the overfitting; §5 changes the objective, ships `agents/unit-12.py`, and checks it on a validation track and then, once, on a **test** track; §6 profiles `step()`; §7 is determinism, rounding traps and packaging.

## 1. The objective, and its noise

A **black-box objective** is a function you can evaluate but not look inside.
Here it takes a setting (some module constants) and returns race times.
This unit's objective:

- race the setting on each of the five **training** tracks (each with its own line and measured grip, Units 8 and 9), at race seed r;
- the **per-seed sum** is the total of those five race times;
- a setting's **evaluation** is its per-seed sums at seeds 0 and 1: ten races;
- if **any** of its races is not clean (Unit 9's rule: finished, 0 collisions, at most 2 m from its own line), the setting is **rejected**, whatever its time;
- a candidate replaces the current best setting, the **incumbent**, only when Unit 1's rule says so: `noise_aware_better(candidate's sums, incumbent's sums)`, the gap between the medians larger than either spread.

How big is the noise?
The probe record of plan `roar-012` (p1) raced `agents/unit-11.py` on every training track, at ten race seeds:

In [ ]:
# The probe record of plan roar-012 (p1): agents/unit-11.py, the nominal car, race seeds 0-4 (one track) or 0-9 (sums).
P1_THREE_LAPS = {  # track: (median time in s, spread over seeds 0-4 in s)
    "test-circuit": (279.65, 0.05), "test-circuit-seed1": (276.25, 0.10), "test-circuit-seed2": (275.55, 0.10),
    "test-circuit-seed3": (275.65, 0.15), "test-circuit-seed4": (275.90, 0.15), "test-circuit-seed2026": (282.00, 0.15),
}
SUMS_ONE_LAP = [476.80, 476.70, 477.00, 476.70, 476.60, 476.95, 476.55, 476.90, 476.65, 476.95]  # seeds 0-9
SUMS_THREE_LAPS = [1383.25, 1383.05, 1383.25, 1382.85, 1382.95, 1383.45, 1382.75, 1383.15, 1383.15, 1383.55]
CPU_SECONDS = {1: 2.0, 3: 5.8}  # about this much computer time per race

for name, (median, spread) in P1_THREE_LAPS.items():
    print(f"{name:22s} 3 laps: median {median:.2f} s, spread {spread:.2f} s")
spread_one = max(SUMS_ONE_LAP) - min(SUMS_ONE_LAP)
spread_three = max(SUMS_THREE_LAPS) - min(SUMS_THREE_LAPS)
print(f"the five-track sum over ten seeds: 1 lap spread {spread_one:.2f} s, 3 laps spread {spread_three:.2f} s")
print(f"the hook's 0.3 s: the 1-lap sum at seeds 0-2 spreads {max(SUMS_ONE_LAP[:3]) - min(SUMS_ONE_LAP[:3]):.2f} s")
assert abs(spread_one - 0.45) < 1e-9 and abs(spread_three - 0.80) < 1e-9

# Unit 1's rule on evaluations from the record (per-seed sums at seeds 0 and 1, 3 laps):
unit11_sums, speed_sums = [1383.25, 1383.05], [1334.35, 1334.15]  # premise U12.4
print("the speed setting beats unit-11.py on the training family:", noise_aware_better(speed_sums, unit11_sums))
print("unit-11.py at seeds 2 and 3 beats unit-11.py at seeds 0 and 1:", noise_aware_better(SUMS_THREE_LAPS[2:4], unit11_sums))
assert noise_aware_better(speed_sums, unit11_sums) and not noise_aware_better(SUMS_THREE_LAPS[2:4], unit11_sums)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(range(10), SUMS_ONE_LAP, "o-")
ax.axhline(np.median(SUMS_ONE_LAP), color="grey", linestyle="--", label="median")
ax.set_xlabel("race seed")
ax.set_ylabel("sum of five 1-lap times (s)")
ax.set_title("the objective's noise: unit-11.py, the same file at ten race seeds")
ax.legend()
plt.show()

One track's 3-lap time moves by 0.05 to 0.15 s from seed to seed; the five-track sum moves by 0.45 s over ten seeds at 1 lap and 0.80 s at 3 laps.
That is the "0.3 s of noise" in the hook (the 1-lap sum over three seeds).
The noise comes from the **race seed**, racekit's draw of the car's noise; the same seed gives the same race, row for row (§7).
A difference smaller than the spread is not evidence of anything, which is why the objective is compared by Unit 1's rule and never by one race.

**One lap or three?**
A 1-lap race costs about 2.0 s of computer time, a 3-lap race about 5.8.
Over 46 clean settings the record ranked the settings by 1-lap and by 3-lap time in almost the same order (a rank correlation of 0.96, where 1 is the same order).
But 1 lap can miss a **late crash**: `MPC_Q_E = 0.6` was clean in all 15 of its 1-lap races, and at 3 laps it crashed on one of the five tracks (the five-track sum 226 s slower at seed 0).
So this unit **searches at 1 lap and checks at 3**.

## 2. The knob map

Which constants do anything?
The record (p2) moved 17 of them one at a time, a few values each, and raced every value on the five training tracks, 1 lap, race seeds 0 to 2 (15 races a value).
Here is the change in the five-track sum (the median over the three seeds, against `unit-11.py`'s 476.83 s), with the noise at 0.30 s:

In [ ]:
NOISE = 0.30  # s: the 1-lap five-track sum's spread over seeds 0-2 (p2's baseline)
# p2: knob -> {value: change in the 1-lap five-track sum (s)}; "identical" = every race row for row the same as
# unit-11.py's; NOT_CLEAN lists the values with a race that was not clean.
P2 = {
    "MU_FRACTION": {0.50: 8.87, 0.52: 4.25, 0.56: -4.10, 0.58: -7.98, 0.60: -11.73},
    "A_DEC": {4.0: 7.18, 4.5: 3.05, 5.5: -2.38, 6.0: -4.10},
    "A_ACC": {3.0: 15.30, 3.5: 6.35, 4.5: -2.88, 5.0: -0.38, 6.0: 0.35},
    "SPEED_KP": {0.3: -9.35, 0.8: 6.10, 1.5: 11.32},
    "SPEED_KI": {0.0: 18.70, 0.05: 2.07, 0.2: -1.13},
    "TARGET_SPEED": {40.0: 8.18, 50.0: -1.43, 60.0: -1.43},
    "MPC_Q_E": {0.1: 0.58, 0.6: -0.20, 1.5: 25.10},
    "MPC_Q_PSI": {5.0: 90.00, 40.0: 0.22, 100.0: 0.87},
    "PROFILE_LEAD_S": {0.1: 0.82, 0.2: -0.05, 0.4: -0.07, 0.5: 0.10},
    "LEAD_MIN_M": {1.0: "identical", 4.0: -0.15, 8.0: -0.77},
    "MPC_R_DU": {300.0: -0.07, 3000.0: 0.05},
    "MPC_HORIZON": {20: 0.05, 30: 0.02, 60: -0.02},
    "REACHED_RADIUS": {2.0: -0.10, 4.0: 0.13},
    "A_THROTTLE": {5.0: 0.0, 7.0: 0.0},
    "FALLBACK_SCALE": {0.8: "identical", 0.92: "identical"},
    "CURVATURE_HORIZON": {10: "identical", 50: "identical"},
    "PREDICT_TICKS": {5: "identical"},
}
NOT_CLEAN = {("MPC_Q_E", 1.5): "13/15 clean", ("MPC_Q_PSI", 5.0): "8/15 clean", ("PROFILE_LEAD_S", 0.1): "13/15 clean"}


def kind(knob):
    # The knob's largest effect among its clean values, against the noise.
    effects = [abs(delta) for value, delta in P2[knob].items() if delta != "identical" and (knob, value) not in NOT_CLEAN]
    if not effects:
        return "dead: every race identical", 0.0
    biggest = max(effects)
    if biggest > 5 * NOISE:
        return "big: more than 5 x the noise", biggest
    if biggest > NOISE:
        return "beyond the noise, under 1 s", biggest
    return "within the noise", biggest


kinds = {knob: kind(knob) for knob in P2}
for knob, (label, biggest) in sorted(kinds.items(), key=lambda item: -item[1][1]):
    crashes = [f"{value} ({NOT_CLEAN[(name, value)]})" for name, value in NOT_CLEAN if name == knob]
    print(f"{knob:18s} {label:30s} largest clean effect {biggest:5.2f} s"
          f"{'; NOT clean at ' + ', '.join(crashes) if crashes else ''}")
big = [knob for knob in P2 if kinds[knob][0] == "big: more than 5 x the noise"]
assert sorted(big) == sorted(["MU_FRACTION", "A_DEC", "A_ACC", "SPEED_KP", "SPEED_KI", "TARGET_SPEED"])

fig, ax = plt.subplots(figsize=(10, 4))
order = sorted(P2, key=lambda knob: -kinds[knob][1])
for rank, knob in enumerate(order):
    ax.semilogy([rank], [max(kinds[knob][1], 0.01)], "o", label=knob)
ax.axhline(NOISE, color="grey", linestyle="--")
ax.axhline(5 * NOISE, color="black", linestyle=":")
ax.set_xlabel("knob, largest effect first (dashed: the noise, 0.30 s; dotted: 5 x the noise)")
ax.set_ylabel("largest clean effect (s)")
ax.set_title("p2: one knob at a time, five training tracks, 1 lap (dead knobs drawn at 0.01 s)")
ax.legend(ncol=3, fontsize=8)
plt.show()

Six knobs move a clean race by more than five times the noise, by 7 to 19 s: `MU_FRACTION`, `A_DEC`, `A_ACC`, `SPEED_KP`, `SPEED_KI` and `TARGET_SPEED`.
Three more move it by less than a second, yet beyond the noise: `LEAD_MIN_M = 8`, `MPC_Q_E = 0.1` and `MPC_Q_PSI = 100`.
Two of those three also **act by crashing** at other values (`MPC_Q_E = 1.5`, `MPC_Q_PSI = 5`), and so does `PROFILE_LEAD_S = 0.1`.
`MPC_R_DU`, `MPC_HORIZON` and `REACHED_RADIUS` stay inside the noise, and `A_THROTTLE` moved no race time at all.
And `FALLBACK_SCALE`, `CURVATURE_HORIZON`, `PREDICT_TICKS` and `LEAD_MIN_M = 1` do **nothing**: every race row for row the same.

**All 48, classified.**
p2 tried 17.
The file has 48, and you can classify every one by reading the agent's code: where is it read, and is that code run in a clean race in the shipped mode (`STEERING = "mpc"`, `SPEED_MODE = "profile"`, no respawn)?

- **acts in a clean race**: a number read on the path a clean race runs, so changing it can change the race;
- **respawn-only**: a number read only after a respawn (the fallback's settings);
- **not a knob here**: a mode switch (it chooses which code runs), a number only an instrument reads (nothing acts on it), or a number only another mode reads.

In [ ]:
ACTS, RESPAWN, NOT_A_KNOB = "acts in a clean race", "respawn-only", "not a knob here"
KNOB_MAP = {  # name: (class, where it is read)
    "REACHED_RADIUS": (ACTS, "advance_index: when a waypoint counts as reached"),
    "TARGET_SPEED": (ACTS, "corner_cap: the profile's top speed"),
    "CROSS_TRACK_REACH": (ACTS, "measure: the window the cross-track error is searched in"),
    "CURVATURE_HORIZON": (NOT_A_KNOB, "curvature_ahead, read only by SPEED_MODE 'cap'"),
    "WHEELBASE": (ACTS, "the MPC's curvature to wheel angle (a car parameter, Unit 3)"),
    "MAX_STEER": (ACTS, "the steer command's scale (measured, Unit 8)"),
    "A_THROTTLE": (ACTS, "mpc: caps how fast the reference speed may rise (and predict)"),
    "A_BRAKE": (NOT_A_KNOB, "predict only: an instrument nothing reads"),
    "C_DRAG": (NOT_A_KNOB, "predict only"),
    "CONTROL_DT": (ACTS, "the MPC's and the speed loop's tick (the race's rule, 0.05 s)"),
    "PREDICT_TICKS": (NOT_A_KNOB, "predict only"),
    "SPEED_KP": (ACTS, "the speed loop's proportional gain"),
    "SPEED_KI": (ACTS, "the speed loop's integral gain"),
    "K_E": (NOT_A_KNOB, "Unit 4's PID steering: STEERING 'pid' only"),
    "K_HEAD": (NOT_A_KNOB, "STEERING 'pid' only"),
    "FEEDFORWARD": (NOT_A_KNOB, "STEERING 'pid' only"),
    "STEERING": (NOT_A_KNOB, "a mode switch: which steering law runs"),
    "PP_K": (RESPAWN, "pure pursuit, the fallback's steering"),
    "PP_MIN": (RESPAWN, "pure pursuit"),
    "PP_MAX": (RESPAWN, "pure pursuit"),
    "ST_K": (NOT_A_KNOB, "Stanley: STEERING 'stanley' only"),
    "SPEED_MODE": (NOT_A_KNOB, "a mode switch: profile, cap or flat"),
    "A_LAT": (NOT_A_KNOB, "only without MU_G, or in cap mode"),
    "A_DEC": (ACTS, "the speed profile's braking (the backward pass)"),
    "A_ACC": (ACTS, "the speed profile's acceleration (the forward pass)"),
    "PROFILE_LEAD_S": (ACTS, "profile_goal: how far ahead the speed goal is read"),
    "LEAD_MIN_M": (ACTS, "profile_goal: ... but at least this far"),
    "GRIP": (ACTS, "lateral_limit: one multiplier per grip section"),
    "MU_G": (ACTS, "lateral_limit: the measured grip (a measurement, per track)"),
    "MU_FRACTION": (ACTS, "lateral_limit: the planner's share of the measured grip"),
    "LINE_STEP": (ACTS, "the line's spline spacing (tied to LINE_CONTROL's length)"),
    "LINE_CONTROL": (ACTS, "the racing line (solved, per track)"),
    "MPC_HORIZON": (ACTS, "the MPC's horizon"),
    "MPC_Q_E": (ACTS, "the MPC's cost on the line error"),
    "MPC_Q_PSI": (ACTS, "the MPC's cost on the heading error"),
    "MPC_R_DU": (ACTS, "the MPC's cost on changing the command"),
    "MPC_LAG": (ACTS, "the lag the MPC models"),
    "MPC_BUDGET_MS": (ACTS, "mpc_command: a call over budget halves the horizon (timing on only)"),
    "MPC_MIN_HORIZON": (NOT_A_KNOB, "read only after an over-budget call: never with the timing off"),
    "RESPAWN_IMPULSE": (RESPAWN, "respawned(): matters only when an impulse arrives"),
    "REFRACTORY_TICKS": (RESPAWN, "respawned(): after an impulse"),
    "FALLBACK_STEERING": (NOT_A_KNOB, "a mode switch: the fallback's steering law"),
    "FALLBACK_SCALE": (RESPAWN, "fall_back(): the lateral limit's cut"),
    "WARN_BETA_DEG": (NOT_A_KNOB, "warnings: counted, never acted on"),
    "WARN_ERROR_M": (NOT_A_KNOB, "warnings"),
    "WARN_MIN_SPEED": (NOT_A_KNOB, "warnings"),
    "WARN_FALLBACK": (NOT_A_KNOB, "a mode switch, for testing only"),
    "STOP_SUBSTEPS": (NOT_A_KNOB, "predict only"),
}
assert set(KNOB_MAP) == set(knobs_in_file)  # every constant of the file, and nothing else
counts = {label: [name for name in knobs_in_file if KNOB_MAP[name][0] == label] for label in (ACTS, RESPAWN, NOT_A_KNOB)}
for label, names in counts.items():
    print(f"{label} ({len(names)}): {', '.join(names)}")
print(f"tunable numbers: {len(counts[ACTS])} + {len(counts[RESPAWN])} = {len(counts[ACTS]) + len(counts[RESPAWN])}")
assert [len(names) for names in counts.values()] == [24, 6, 18]

24 constants act in a clean race and 6 more only after a respawn: **30 tunable numbers**.
The hook's "about 30" is design 013's round number, and it happens to land on this count; a different line between "a knob" and "not a knob" gives a different count.
Some of the 24 are not things you would tune at all: `WHEELBASE` and `MAX_STEER` are measured car parameters, `CONTROL_DT` is the race's rule, `MU_G` and `LINE_CONTROL` are measurements and solutions per track, and `LINE_STEP` must match `LINE_CONTROL`'s length.
So the knobs worth searching are fewer still, and p2 says which: the big six.

One constant needs more than reading: `A_THROTTLE` is read by `predict()` (dead) **and** by `mpc()`, which caps how fast the reference speed may rise.
In p2 it moved no race time at all.
Is it dead?
Race it, and compare the **telemetry**, not only the time.
Here, on `test-circuit`, 1 lap, race seed 0: the shipped file, a knob the shipped mode never reads (`FALLBACK_SCALE = 0.5`), a big one (`MU_FRACTION = 0.60`) and `A_THROTTLE = 5.0`:

In [ ]:
demos = {
    "shipped": race({}),
    "FALLBACK_SCALE = 0.5": race({"FALLBACK_SCALE": 0.5}),
    "MU_FRACTION = 0.60": race({"MU_FRACTION": 0.60}),
    "A_THROTTLE = 5.0": race({"A_THROTTLE": 5.0}),
}
shipped = demos["shipped"]
for label, result in demos.items():
    print(f"{label:22s} {result.total_time:6.2f} s, clean {clean(result)}, the same race as the shipped file: "
          f"{same_race(result, shipped)}")
assert clean(shipped) and abs(shipped.total_time - 96.40) <= TICK  # U12.2
assert same_race(demos["FALLBACK_SCALE = 0.5"], shipped)  # U12.2: a dead knob
assert abs(demos["MU_FRACTION = 0.60"].total_time - 93.95) <= TICK  # U12.2: a big one
assert demos["A_THROTTLE = 5.0"].total_time == shipped.total_time and not same_race(demos["A_THROTTLE = 5.0"], shipped)

`FALLBACK_SCALE = 0.5` is the same race as the shipped file, every row: without a respawn, `fall_back()` never runs.
`MU_FRACTION = 0.60` is 2.45 s faster in one lap of one track.
And `A_THROTTLE = 5.0` finishes in the same 96.40 s, but it is **a different race**: the rows differ, because `mpc()` planned with a different reference speed.
So `A_THROTTLE` acts in a clean race (it is in the 24), even though it moved no race time in p2: "dead" means the same telemetry, and only a telemetry comparison can say so.
A search that spends races on a dead knob learns nothing; one that spends them on a knob that acts without moving the time learns only the noise.

## 3. Searching

**Coordinate search.**
Take one knob at a time.
Try one grid step up and one grid step down; if the better of the two beats the incumbent (Unit 1's rule, and clean), take it, and keep stepping that way while each step still wins; then go on to the next knob.
Stop when every knob has been tried without a gain, or when the race budget runs out.

Every step is **snapped to the grid** with `round(x, 2)`: 0.54 + 0.02 + 0.02 is `0.5800000000000001` in floating point, a different number from 0.58, and §7 shows that it is a different race.

The search below runs on the three knobs the record's speed search moved (`MU_FRACTION`, `A_DEC`, `A_ACC`), from `unit-11.py`'s values, at **1 lap** and race seeds 0 and 1: ten races an evaluation.
Its `step()` runs exactly one evaluation, so the search can stop after any evaluation and go on later.
This cell runs the first 20 races (about 45 s); the next cell runs 20 more.

In [ ]:
START = {"MU_FRACTION": 0.54, "A_DEC": 5.0, "A_ACC": 4.0}  # agents/unit-11.py's values
GRID = {"MU_FRACTION": (0.48, 0.66, 0.02), "A_DEC": (4.0, 8.0, 0.5), "A_ACC": (3.0, 6.0, 0.5)}  # (low, high, step)
SEARCH_SEEDS = (0, 1)
evaluations = {}  # every evaluation so far, by setting: each setting is raced once


def evaluate(setting):
    # Ten races: the five training tracks at race seeds 0 and 1, 1 lap; the per-seed sums; clean if all ten are.
    key = tuple(sorted(setting.items()))
    if key not in evaluations:
        results = {(track, seed): race(setting, track, seed) for seed in SEARCH_SEEDS for track in TRAINING}
        sums = [sum(results[(track, seed)].total_time for track in TRAINING) for seed in SEARCH_SEEDS]
        evaluations[key] = {"setting": dict(setting), "sums": sums, "results": results,
                            "clean": all(clean(result) for result in results.values()), "races": 10 * (len(evaluations) + 1)}
    return evaluations[key]


def score(evaluation):
    # The median of the per-seed sums; a setting that is not clean scores infinity (rejected).
    return float(np.median(evaluation["sums"])) if evaluation["clean"] else math.inf


def beats(candidate, incumbent):
    return candidate["clean"] and noise_aware_better(candidate["sums"], incumbent["sums"])


def neighbour(setting, knob, direction):
    # One grid step of one knob, kept inside the box and snapped to the grid.
    low, high, step = GRID[knob]
    return {**setting, knob: round(min(max(setting[knob] + direction * step, low), high), 2)}


class CoordinateSearch:
    def __init__(self, start, knobs):
        self.start, self.knobs = dict(start), knobs
        self.incumbent, self.up, self.direction = None, None, 0
        self.k = 0  # the knob being tried
        self.phase = "start"  # start, up, down, onward or done
        self.without_gain = 0  # knobs tried since the last step taken
        self.taken = []  # (races so far, the setting taken, its score)

    def next_knob(self):
        self.k = (self.k + 1) % len(self.knobs)
        self.without_gain += 1
        self.phase = "done" if self.without_gain >= len(self.knobs) else "up"

    def take(self, evaluation):
        self.incumbent = evaluation
        self.taken.append((evaluation["races"], evaluation["setting"], score(evaluation)))

    def step(self):
        # One evaluation (or none, when the step is cached or the search is done).
        knob = self.knobs[self.k]
        if self.phase == "start":
            self.take(evaluate(self.start))
            self.phase = "up"
        elif self.phase == "up":
            self.up = evaluate(neighbour(self.incumbent["setting"], knob, +1))
            self.phase = "down"
        elif self.phase == "down":
            down = evaluate(neighbour(self.incumbent["setting"], knob, -1))
            better, direction = (self.up, +1) if score(self.up) <= score(down) else (down, -1)
            if beats(better, self.incumbent):
                self.take(better)
                self.direction, self.without_gain, self.phase = direction, 0, "onward"
            else:
                self.next_knob()
        elif self.phase == "onward":
            candidate = evaluate(neighbour(self.incumbent["setting"], knob, self.direction))
            if beats(candidate, self.incumbent):
                self.take(candidate)
            else:
                self.next_knob()


def show(search):
    for evaluation in evaluations.values():
        print(f"after {evaluation['races']:2d} races: {evaluation['setting']} sums {evaluation['sums'][0]:.2f} / "
              f"{evaluation['sums'][1]:.2f} s, clean {evaluation['clean']}, score {score(evaluation):.2f} s")
    print(f"incumbent: {search.incumbent['setting']}, score {score(search.incumbent):.2f} s; next: {search.phase} "
          f"{search.knobs[search.k]}")


search = CoordinateSearch(START, ("MU_FRACTION", "A_DEC", "A_ACC"))
while 10 * len(evaluations) < 20 and search.phase != "done":
    search.step()
show(search)
first = evaluations[tuple(sorted(START.items()))]
assert abs(score(first) - 476.75) <= TICK  # p8: the starting objective
assert search.incumbent is first and abs(score(search.up) - 472.62) <= TICK  # MU 0.56 raced, not yet taken

Twenty races: the start, `unit-11.py`'s own values (476.75 s, the two sums 0.10 s apart), and one step up in `MU_FRACTION`, 0.56, about 4.1 s faster.
It has not been taken yet: the search still has to try 0.52 before it chooses.
Its incumbent, `search`, and every evaluation so far (`evaluations`) stay in memory, so the next cell carries on exactly where this one stopped.

In [ ]:
while 10 * len(evaluations) < 40 and search.phase != "done":
    search.step()
show(search)
for races, setting, value in search.taken:
    print(f"taken after {races} races: {setting}, {value:.2f} s")
assert len(evaluations) == 4 and search.phase == "onward"  # the budget ran out, the search had not finished
assert search.incumbent["setting"] == {"MU_FRACTION": 0.58, "A_DEC": 5.0, "A_ACC": 4.0}
assert abs(score(search.incumbent) - 468.77) <= TICK  # p8: 476.75 -> 472.62 (30 races) -> 468.77

# The same seed is the same race: the search's own race of the shipped values on test-circuit, seed 0, and §2's.
print("the search's test-circuit seed-0 race is §2's shipped race, row for row:",
      same_race(first["results"][("test-circuit", 0)], shipped))
assert same_race(first["results"][("test-circuit", 0)], shipped)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot([e["races"] for e in evaluations.values()], [score(e) for e in evaluations.values()], "o", label="every evaluation")
ax.step([races for races, _, _ in search.taken] + [40], [value for _, _, value in search.taken] + [score(search.incumbent)],
        where="post", label="the incumbent")
ax.set_xlabel("races")
ax.set_ylabel("score: median per-seed sum (s, 5 laps)")
ax.set_title("coordinate search, live: MU_FRACTION 0.54, 0.56, 0.52, 0.58 (1 lap, seeds 0 and 1)")
ax.legend()
plt.show()

After 40 races the search has tried 0.52 (slower), taken 0.56 (472.62 s), and gone on to 0.58 (468.77 s): 8.0 s faster over the five laps.
It **stopped on its budget, not because it had finished**: it was about to try 0.60.
The record (p7) let the same search run 50 races; it reached 0.60 (465.02 s), and that setting is clean at 3 laps on all ten training races.
And the record's full search (p3: six knobs, 3 laps, 200 races, about 1,100 s of computer time) is the hook's: 1383.15 → 1334.25 s, at `MU_FRACTION` 0.60, `A_DEC` 5.5 and `A_ACC` 4.5.
The live search above is that search at a size a notebook can run.

The search's first evaluation re-raced `test-circuit` at seed 0 with the shipped values, the race §2 already ran: it is the same race, row for row.

**Random search.**
Draw a whole setting at random inside the box, every knob at once, evaluate it, and keep it if it beats the incumbent.
It needs no idea of which knob matters, and it wastes races on every part of the box that does not.
A random draw uses numpy's generator: `rng = np.random.default_rng(seed)` makes one (seeded, so the draws are the same every time), and `rng.uniform(low, high)` draws a number evenly between `low` and `high`.

**Evolution strategy, (1+λ)-ES.**
Keep one parent, the incumbent.
Each generation draws λ children near it: every knob moved by a normal (Gaussian) draw, `rng.normal(0.0, sigma)`, mean 0 and standard deviation `sigma` (about two thirds of the draws fall within ±`sigma`).
The best child replaces the parent if it beats it.
The step size adapts: after a success, `sigma` grows (×1.5: steps that work may work bigger); after a failure it shrinks (×0.8).

The record (p3) ran both on the same six knobs, objective and budget as the hook's coordinate search, and here are a few draws of each, live (no races):

In [ ]:
# The probe record of plan roar-012 (p3): (races so far, the incumbent's objective in s) at each step taken, 200 races each.
P3 = {
    "coordinate": P3_COORDINATE,
    "random (batches of 6)": [(10, 1383.15), (130, 1381.53), (200, 1381.53)],
    "(1+4)-ES": [(10, 1383.15), (50, 1360.28), (170, 1357.17), (200, 1357.17)],
}
# Random search drew two settings that beat its own incumbent but were not clean (0 collisions):
P3_RANDOM_NOT_CLEAN = [(1354.45, 3.13), (1350.75, 2.51)]  # (objective in s, max line error in m)
for method, path in P3.items():
    print(f"{method:22s} {path[0][1]:.2f} -> {path[-1][1]:.2f} s (gain {path[0][1] - path[-1][1]:5.2f} s)")
for value, error in P3_RANDOM_NOT_CLEAN:
    print(f"random, not clean: {value:.2f} s at {error:.2f} m from its line: counted at its time, it would have won")

rng = np.random.default_rng(12)
print("five random settings in the box (each rounded to 2 decimals):")
for _ in range(5):
    print("   ", {knob: round(rng.uniform(low, high), 2) for knob, (low, high, step) in GRID.items()})
sigma = {knob: step for knob, (low, high, step) in GRID.items()}  # one grid step each
parent = search.incumbent["setting"]
print(f"two ES children of {parent}, sigma {sigma}:")
for _ in range(2):
    print("   ", {knob: round(min(max(parent[knob] + rng.normal(0.0, sigma[knob]), GRID[knob][0]), GRID[knob][1]), 2)
                  for knob in parent})
print("sigma after a success:", {knob: round(1.5 * value, 3) for knob, value in sigma.items()},
      "after a failure:", {knob: round(0.8 * value, 3) for knob, value in sigma.items()})

fig, ax = plt.subplots(figsize=(8, 3.5))
for method, path in P3.items():
    races, objective = zip(*path)
    ax.step(races, objective, where="post", label=method)
ax.set_xlabel("races (3 laps each)")
ax.set_ylabel("incumbent objective (s, 15 laps)")
ax.set_title("p3: three searches, the same six knobs and 200 races")
ax.legend()
plt.show()

On the same budget, coordinate search gained 48.9 s, the evolution strategy 26.0 s and random search 1.6 s.
Most of a six-knob box is slower than `unit-11.py` or not clean: a random draw lands there most of the time, a coordinate step only moves where a step has already paid.
That is a finding about **this** box and objective, not a law: the knobs here act almost independently and smoothly, which is the best case for one-knob-at-a-time.

Random search also shows why a not-clean race must **reject** a setting and not just count its time.
Two of its draws were faster than its incumbent and finished with 0 collisions, but 2.5 and 3.1 m off their lines.
Counted at their times they would have been taken: an agent that wins by leaving its line, on one car, is one perturbation away from a crash (Unit 11).

**Optional: a taste of policy gradient.**
Reinforcement learning tunes a **policy** instead of a setting.
For one knob, take the policy to be a normal distribution of values, mean μ and standard deviation σ: every "race" draws its knob value x from it.
To move μ toward faster races, draw N values x₁ … x_N, race each (times T₁ … T_N), and update

  μ ← μ + α · mean over the draws of (b − Tᵢ) · (xᵢ − μ) / σ²,

where the **baseline** b is the mean of the N race times.
A draw faster than the baseline (b − Tᵢ > 0) pulls μ toward itself; a slower one pushes μ away.
Without the baseline every term would carry the whole race time, about 476 s, and the update would be almost all noise; with it, each term carries only how much better or worse that draw was.
α is a step size (here in (m/s²)² per second).

That is the whole method: `rng.normal` and arithmetic.
To see it work without spending races, give it a **pretend race**: p2's measured `A_ACC` curve (the 1-lap five-track sum at six values, joined by straight lines) plus noise drawn with the objective's measured spread.
It is a stand-in, made from the record, and it is labelled as one everywhere below.
Policy gradient and coordinate search each get **20 pretend races**:

In [ ]:
A_ACC_CURVE = {3.0: 15.30, 3.5: 6.35, 4.0: 0.0, 4.5: -2.88, 5.0: -0.38, 6.0: 0.35}  # p2: change in the 1-lap sum (s)
BASE_SUM, NOISE_SD = 476.83, 0.16  # s: unit-11.py's 1-lap sum, and its standard deviation over ten seeds (p1)
BUDGET = 20


def curve(a_acc):
    # p2's measured change at a_acc, with straight lines between the measured values (kept inside 3.0 to 6.0).
    knots = sorted(A_ACC_CURVE)
    a_acc = min(max(a_acc, knots[0]), knots[-1])
    for low, high in zip(knots, knots[1:]):
        if a_acc <= high:
            return A_ACC_CURVE[low] + (a_acc - low) / (high - low) * (A_ACC_CURVE[high] - A_ACC_CURVE[low])


def pretend_race(a_acc, noise):
    # NOT a race: p2's measured curve, plus noise.
    return BASE_SUM + curve(a_acc) + noise.normal(0.0, NOISE_SD)


# Policy gradient: mean mu, standard deviation sigma, N = 4 draws an update, 5 updates.
noise, policy = np.random.default_rng(1), np.random.default_rng(2)
mu, sigma_pg, alpha, draws = 4.0, 0.25, 0.04, 4
pg_path = [mu]
for update in range(BUDGET // draws):
    x = [mu + policy.normal(0.0, sigma_pg) for _ in range(draws)]
    times = [pretend_race(value, noise) for value in x]
    baseline = float(np.mean(times))
    gradient = float(np.mean([(baseline - t) * (value - mu) / sigma_pg ** 2 for t, value in zip(times, x)]))
    mu = mu + alpha * gradient
    pg_path.append(mu)
print("policy gradient, mu after each update:", [round(value, 2) for value in pg_path])

# Coordinate search on the same knob and budget: every setting twice (two "seeds"), Unit 1's rule, grid 0.5;
# step up while a step wins, then down.
noise = np.random.default_rng(1)
pretend = {}


def pretend_evaluate(a_acc):
    if a_acc not in pretend:
        pretend[a_acc] = [pretend_race(a_acc, noise) for _ in range(2)]
    return pretend[a_acc]


best = 4.0
pretend_evaluate(best)
for direction in (+1, -1):
    while 2 * len(pretend) + 2 <= BUDGET:
        candidate = round(best + direction * 0.5, 2)
        if not noise_aware_better(pretend_evaluate(candidate), pretend_evaluate(best)):
            break
        best = candidate
print(f"coordinate search: A_ACC {best} after {2 * len(pretend)} pretend races; tried {sorted(pretend)}")
print(f"p2's curve (without noise) at coordinate search's answer {curve(best):+.2f} s, at policy gradient's mean "
      f"{curve(mu):+.2f} s")
assert best == 4.5 and 2 * len(pretend) <= BUDGET and abs(mu - 4.5) < 0.1

Coordinate search found 4.5, the best value on its grid, and stopped after 6 of its 20 pretend races.
Policy gradient spent all 20 and its mean ended near 4.5 too, after a first update that jumped straight there and a few that wandered: every update rests on 4 noisy draws, and its answer is a distribution, not a setting, so you still have to race its mean.
On one smooth knob, plain search wins easily; policy gradient earns its keep where a policy has to choose among many actions in many situations, not on one constant.
And a crash would be just a very long race time to it: nothing in the update **rejects** a not-clean draw.
Challenge 10 races it for real, on the same race budget as a coordinate search.

## 4. Overfitting

The hook's search was tuned on five tracks and failed on a sixth.
Push it further: tune on **one** track, with more knobs.
The record (p5) ran a coordinate search on `test-circuit` alone (3 laps, race seeds 0 and 1, 60 races) over the six global knobs **plus** the six per-section grip knobs `GRIP` (Unit 7: one multiplier per grip section, box 0.7 to 1.5, step 0.05).
It found `MU_FRACTION` 0.64, `A_DEC` 5.5, and `GRIP` 1.2 on section 0 and 1.05 on section 2.
Then the record raced that setting on the other tracks (3 laps, race seeds 0 to 2):

In [ ]:
# p5: the one-track setting, written exactly as the search produced it (grid steps added one at a time, §7):
OVERFIT = {"MU_FRACTION": 0.6400000000000001, "A_DEC": 5.5, "GRIP": (1.2000000000000002, 1.0, 1.05, 1.0, 1.0, 1.0)}
P5 = {  # track: (unit-11.py median, the one-track setting's median (s), its clean races of 3, its collisions)
    "test-circuit": (279.65, 265.55, 3, 0),
    "test-circuit-seed1": (276.30, 328.60, 0, 5),
    "test-circuit-seed2": (275.55, 389.75, 0, 3),
    "test-circuit-seed3": (275.75, 389.75, 0, 3),
    "test-circuit-seed4": (275.95, 442.95, 0, 6),
    "test-circuit-seed2026": (282.00, 369.35, 0, 6),
}
for name, (shipped_time, tuned_time, n_clean, collisions) in P5.items():
    print(f"{name:22s} unit-11.py {shipped_time:.2f} s | one-track setting {tuned_time:.2f} s, {n_clean}/3 clean, "
          f"{collisions} collisions")

# One validation race, live: the one-track setting on test-circuit-seed2026, 3 laps, race seed 0.
overfit = race(OVERFIT, VALIDATION, seed=0, laps=3)
print(f"live, {VALIDATION}, seed 0: {overfit.total_time:.2f} s, {overfit.collisions} collisions (at "
      f"{', '.join(f'{t:.2f}' for t in overfit.respawn_times)} s), {line_error(overfit):.2f} m off its line, clean {clean(overfit)}")
assert not clean(overfit) and overfit.collisions == 2 and abs(overfit.total_time - 347.20) <= TICK  # U12.3

On the track it was tuned on, the setting is 14.1 s faster than `unit-11.py`, and clean.
On every other track it crashes, in every race: up to 6 collisions in three races, and 87 s slower on the validation track's median.
Live, on `test-circuit-seed2026`, it crashed twice and lost 65 s to `unit-11.py`'s 282.05 s at the same seed.

This is **overfitting**: with enough knobs, a search fits the tracks it is scored on, including everything about them that will not be there next time.
Per-section `GRIP` knobs are the worst case: they describe one track's six sections, and on another track the same six numbers sit on different corners.
The hook's search had fewer knobs and five tracks, and it overfit too, more slowly: the five training tracks, on one car, did not contain the failure.

**Training, validation and test tracks.**
The defence is to keep tracks the search never scores on:

- **training** tracks: the objective races them, as often as it likes;
- a **validation** track: race a setting that is already chosen, to catch overfitting; if the result makes you change the search or the objective, the validation track has **taken part in the choice**;
- a **test** track: raced **once**, at the very end, to measure what you built; no decision may depend on it.

`test-circuit-seed2026` is this unit's validation track.
It already changed a decision: the hook's failure there is why §5 changes the objective.
From then on it cannot be an independent test of the result, so the unit keeps a fresh track for that, `test-circuit-seed2028`, its seed fixed before anyone had looked at it.

## 5. A crash-aware objective, and `agents/unit-12.py`

The speed search failed because its family (five tracks, one car) did not contain the failure.
Unit 11 already has the cars that do: its perturbed family.
So change the **objective**, not the search: to each evaluation's ten training races add `test-circuit` on Unit 11's three lowest-grip training cars, 16, 18 and 8 (race seed (k + r) mod 5 for replicate r), and still reject any setting with a race that is not clean.
That is 16 races an evaluation, and a crash on a low-grip car now rejects the setting.

The record (p4) ran a coordinate search over `MU_FRACTION`, `A_DEC` and `A_ACC` with this objective, 3 laps, 160 races.
Here are all ten of its evaluations:

In [ ]:
# p4 (premise U12.6 re-races the first and fourth): (MU_FRACTION, A_DEC, A_ACC) -> the two per-replicate sums
# of 16 races (s), and the races that were not clean ("" if none).
CRASH_AWARE = {
    (0.54, 5.0, 4.0): ((2224.90, 2224.35), ""),  # agents/unit-11.py
    (0.56, 5.0, 4.0): ((2411.70, 2205.10), "car 8"),
    (0.52, 5.0, 4.0): ((2245.50, 2245.25), ""),
    (0.54, 5.5, 4.0): ((2215.35, 2215.25), ""),  # agents/unit-12.py
    (0.54, 4.5, 4.0): ((2238.75, 2238.90), ""),
    (0.54, 6.0, 4.0): ((2984.10, 2966.65), "cars 16, 18, 8 (both replicates), nominal test-circuit"),
    (0.54, 5.5, 4.5): ((2718.85, 2203.70), "cars 16, 8"),
    (0.54, 5.5, 3.5): ((2246.25, 2246.45), "car 18 (both replicates), car 8"),
    (0.56, 5.5, 4.0): ((2495.40, 2501.40), "cars 8, 16"),
    (0.52, 5.5, 4.0): ((2235.50, 2235.65), "car 18"),
}
STEPS = (0.02, 0.5, 0.5)  # the grid steps of MU_FRACTION, A_DEC, A_ACC
clean_settings = [s for s, (sums, bad) in CRASH_AWARE.items() if not bad]
for s, (sums, bad) in CRASH_AWARE.items():
    print(f"{s}: {sums[0]:.2f} / {sums[1]:.2f} s, {'clean' if not bad else 'NOT clean: ' + bad}")
best_clean = min(clean_settings, key=lambda s: np.median(CRASH_AWARE[s][0]))
print("the best clean setting:", best_clean, "; it beats unit-11.py by Unit 1's rule:",
      noise_aware_better(CRASH_AWARE[best_clean][0], CRASH_AWARE[(0.54, 5.0, 4.0)][0]))


def neighbours(s):
    return [tuple(round(s[i] + d * STEPS[i], 2) if i == axis else s[i] for i in range(3)) for axis in range(3) for d in (+1, -1)]


clean_neighbours = [n for n in neighbours(best_clean) if n in clean_settings]
print("its grid neighbours that are clean:", clean_neighbours)
# Unit 11's margin rule along A_DEC: the highest clean value is the edge; ship one grid step inside it.
edge = max(s[1] for s in clean_settings if s[0] == 0.54 and s[2] == 4.0)
print(f"the A_DEC edge: {edge}; the margin rule ships {edge - 0.5}, which is unit-11.py")
assert best_clean == (0.54, 5.5, 4.0) and clean_neighbours == [(0.54, 5.0, 4.0)] and edge == 5.5

The search took **one** step, `A_DEC` 5.0 → 5.5 (9.32 s faster over the 16 races, against spreads of 0.55 and 0.10 s), and rejected every other: each was not clean on at least one low-grip car.
That is `agents/unit-12.py`: `unit-11.py` with `A_DEC = 5.5`, and nothing else changed (§7 checks it).

**5.5 is the edge.**
Its only clean grid neighbour is `unit-11.py` itself.
The next step, `A_DEC = 6.0`, crashes on all six low-grip races.
Unit 11's **margin rule** (ship one grid step inside the highest clean value) would ship 5.0: `unit-11.py`, and no tuned agent at all.
This unit ships the edge **on purpose**, to measure what the margin was for.
Exercise 6 asks you to apply the rule yourself and argue which you would race.

What it costs, on Unit 11's perturbed family (`test-circuit`, race seed k mod 5, 3 laps; the probe record and the premise U12.7):

In [ ]:
FAMILY = {  # setting: (training cars 0-23 clean, held-out cars 24-47 clean), of 24 each
    "unit-11.py": (24, 24),
    "the speed setting": (7, 7),
    "unit-12.py": (24, 20),
}
UNIT_12_HELD_OUT_NOT_CLEAN = {24: 2.011, 29: 2.194, 32: 2.170, 35: 2.508}  # car: max line error (m), 0 collisions each
for name, (training, held_out) in FAMILY.items():
    print(f"{name:18s} training cars {training:2d}/24 clean, held-out cars {held_out:2d}/24 clean")
print("unit-12.py's held-out cars that are not clean (each finished, 0 collisions):",
      ", ".join(f"car {k} at {error:.2f} m" for k, error in UNIT_12_HELD_OUT_NOT_CLEAN.items()))

# The validation track (premise U12.5a), 3 laps, race seeds 0-4, both clean in all five:
VALIDATION_TIMES = {"unit-12.py": [280.65, 280.65, 280.60, 280.65, 280.60], "unit-11.py": SEED2026_UNIT_11}
# The test track, test-circuit-seed2028, raced once at the end (premise U12.5; the milestone races it again):
TEST_TIMES = {"unit-12.py": [269.65, 269.65, 269.75, 269.70, 269.65], "unit-11.py": [271.10, 271.05, 271.15, 271.05, 270.95],
              "the speed setting": [261.30, 261.20, 480.45, 261.35, 261.25]}  # the speed setting: 4/5 clean
for track, table in (("validation, seed2026", VALIDATION_TIMES), ("test, seed2028", TEST_TIMES)):
    for name, times in table.items():
        print(f"{track:20s} {name:18s} median {np.median(times):.2f} s, spread {max(times) - min(times):6.2f} s, "
              f"mean {np.mean(times):.2f} s")
for table in (VALIDATION_TIMES, TEST_TIMES):
    print("unit-12.py beats unit-11.py by Unit 1's rule:", noise_aware_better(table["unit-12.py"], table["unit-11.py"]))
    assert noise_aware_better(table["unit-12.py"], table["unit-11.py"])
assert abs(np.mean(TEST_TIMES["the speed setting"]) - 305.11) < 0.01

fig, ax = plt.subplots(figsize=(8, 3.5))
for name, times in TEST_TIMES.items():
    ax.plot(range(5), times, "o-", label=name)
ax.set_xlabel("race seed")
ax.set_ylabel("race time (s, 3 laps)")
ax.set_title("the test track, test-circuit-seed2028 (the premise): one crash costs a whole race")
ax.legend()
plt.show()

**On the validation track**, `unit-12.py` is 1.35 s faster than `unit-11.py` (280.65 against 282.00 s, both clean in all five races, spreads 0.05 and 0.15 s): Unit 1's rule says it is real.
**On the test track**, raced once at the end, it is 1.40 s faster (269.65 against 271.05 s, both clean in all five, spreads 0.10 and 0.20 s): real again.
That is design 013's gate for this unit: the tuned agent beats the hand-tuned one on a held-out track, noise-aware.
The milestone races it.

The speed setting on the test track is the hook's lesson in one row: in its four clean races it is about 8 s faster than `unit-12.py` and nearly 10 s faster than `unit-11.py`, and one crash cost a whole race (480.45 s, Unit 11's respawn economics).
Its mean, about 305 s, is 34 s slower than `unit-11.py`'s 271.

**What 1.35 s costs.**
On Unit 11's 24 held-out cars `unit-12.py` finishes every race with 0 collisions, but 4 of them (cars 24, 29, 32 and 35) run 2.01 to 2.51 m off their lines: over the clean rule's 2 m.
`unit-11.py` is clean on all 24, and on the validation track its largest line error is 1.20 m against `unit-12.py`'s 1.76.
The time was bought with **line margin**, exactly what Unit 11 §4 predicts for an agent with no margin left.
Which agent races is a judgment, and Unit 13 makes it.

**The same overfitting, one level up.**
The crash-aware objective's family is the five tracks and three low-grip cars.
`unit-12.py` is clean on all 24 **training** cars of Unit 11's family, and on 20 of the 24 **held-out** cars.
A crash-aware objective made the held-out failures rarer (17 of 24 for the speed setting, 4 of 24 now) and smaller (line error, no collisions), not impossible: the objective's family still does not contain every failure.
No objective can, and that is why the validation and test tracks exist.

(The rejected `A_DEC = 6.0` was marginal on the nominal car too: in p4's search it was not clean on one of the two nominal `test-circuit` races as well as on every low-grip car.)

## 6. Profiling `step()`

The race allows 50 ms a tick, and `check_submission` asks for `step()`'s 99th percentile under 5 ms (Unit 10).
Searching costs races, and races cost `step()` calls: where does a step's time go?
Python's profiler, `cProfile`, counts every function call and the time spent in it.
`profiler = cProfile.Profile()` makes one, `profiler.enable()` starts it, `profiler.disable()` stops it, and `pstats.Stats(profiler)` reads what it recorded: `sort_stats("cumulative")` orders the functions by the time spent in them **including** everything they call, `sort_stats("tottime")` by the time in their own lines only, and `print_stats(text, n)` prints the first `n` whose file name matches `text`.
One race of `unit-11.py` (timing off) on `test-circuit-seed1`, 1 lap, race seed 0:

In [ ]:
profiled = variant(UNIT_11, **on_track("test-circuit-seed1"), **NO_TIMING)
profiler = cProfile.Profile()
profiler.enable()
profiled_race = run_race(profiled, "test-circuit-seed1", car=carla_like(noise=True), seed=0, laps=1)
profiler.disable()
print(f"{profiled_race.total_time:.2f} s, {len(profiled_race.step_ms)} step() calls; under the profiler, step() p50 "
      f"{np.percentile(profiled_race.step_ms, 50):.3f} ms, p99 {np.percentile(profiled_race.step_ms, 99):.3f} ms")
stats = pstats.Stats(profiler)
stats.sort_stats("cumulative").print_stats(profiled.stem, 12)  # the agent's own functions
stats.sort_stats("tottime").print_stats(8)  # everything, the agent and the simulator

Your numbers depend on the machine and on what else it is doing, so read them as shares, not as times.
The probe record (p6, the same race, measured without the profiler, which slows every call) found:

| | p6 |
|---|---|
| `step()`, the whole tick | p50 0.914 ms, p99 0.995 ms |
| `mpc_command` (the model, the stack, the `lstsq` solve) | 63% of `step()` |
| `predict()` (Unit 3's 1-second rollout) | 19% of `step()` |
| pure pursuit instead of MPC | p50 0.218 ms |
| `PREDICT_TICKS = 1` | p50 0.826 ms |
| `MPC_HORIZON = 20` instead of 40 | p50 0.537 ms, and 3-lap race time 276.35 instead of 276.25 s |

Three findings:

- **The solve dominates.** Most of `mpc_command` is `least_squares_plan`, and most of that is numpy's `lstsq`.
  Make it faster and every tick gets faster.
- **Dead work.** `predict()` costs a fifth of every step, and nothing reads its answer (`predicted_state` is an instrument, §2: `PREDICT_TICKS`, `A_BRAKE`, `C_DRAG` are not knobs here).
  It is harmless at about 1 ms a tick, but it is pure cost.
- **The horizon trade.** Half the horizon nearly halves the step (0.914 → 0.537 ms) for 0.10 s over three laps, two ticks: a cheap trade if the tick budget were tight.
  It is not: `step()` uses about 2% of the 50 ms tick.

The profiler also lists the **simulator** (`racekit/cars.py`): in a race on your machine, the car model costs time too, and none of that time counts against your agent in the official race.

## 7. Determinism, rounding traps, and packaging

**The same seed is the same race.**
You saw it in §3: the search's race of the shipped values on `test-circuit` at seed 0 was §2's race, row for row.
racekit draws all of a race's noise from its seed, and the agent does no arithmetic that depends on the clock (with the timing off).
That is what makes a search possible at all: an evaluation measures the setting, not luck.

**Trap 1: a number written by hand is a different number.**
A float is not its printout.
`repr(x)` writes the shortest text that reads back as exactly `x` (it **round-trips**); any other rounding writes a different float.
Write the same value three ways, with `repr`, to 4 decimals and to 3 decimals: three different floats, the same race time, **three different races**.

**Trap 2: steps add up to a float you did not write.**
0.54 + 3 × 0.02 is `0.6000000000000001`, not 0.6.
The record's speed search added its grid steps without snapping, and that is the value it found.

Both on `test-circuit-seed1`, 1 lap, race seed 0 (trap 1) and `test-circuit`, seed 0 (trap 2, against §2's race at 0.60):

In [ ]:
value = 0.54 * 1.0123456789
texts = {"repr": repr(value), "4 decimals": f"{value:.4f}", "3 decimals": f"{value:.3f}"}
for how, text in texts.items():
    print(f"{how:10s} {text:16s} reads back as the same float: {float(text) == value}")
rounded = {how: race({"MU_FRACTION": float(text)}, "test-circuit-seed1") for how, text in texts.items()}
for how, result in rounded.items():
    print(f"MU_FRACTION texts {how:10s}: {result.total_time:.2f} s")
pairs = [("repr", "4 decimals"), ("repr", "3 decimals"), ("4 decimals", "3 decimals")]
print("any two of them the same race:", [same_race(rounded[a], rounded[b]) for a, b in pairs])
assert float(texts["repr"]) == value and float(texts["4 decimals"]) != value
assert all(abs(result.total_time - 94.95) <= TICK for result in rounded.values())  # p6: the same time ...
assert not any(same_race(rounded[a], rounded[b]) for a, b in pairs)  # ... three different races

stepped = 0.54 + 3 * 0.02
print(f"0.54 + 3 * 0.02 = {stepped!r}; equal to 0.6: {stepped == 0.6}; round(stepped, 2) == 0.6: {round(stepped, 2) == 0.6}")
unsnapped = race({"MU_FRACTION": stepped})
on_grid = demos["MU_FRACTION = 0.60"]
print(f"test-circuit, seed 0: MU_FRACTION {stepped!r} {unsnapped.total_time:.2f} s; 0.6 {on_grid.total_time:.2f} s; "
      f"the same race: {same_race(unsnapped, on_grid)}")
assert unsnapped.total_time == on_grid.total_time and not same_race(unsnapped, on_grid)  # p8, p6: trap 2

# Trap 3: a race time is a whole number of ticks.
ticks = round(279.65 / DT)
print(f"{ticks} ticks x {DT} = {ticks * DT!r}; printed to 2 decimals {ticks * DT:.2f}")
assert ticks * DT != 279.65 and round(ticks * DT, 2) == 279.65

`repr` reads back exactly; 4 and 3 decimals do not.
The three files finish in the same 94.95 s and are three different races: a difference in the fifth significant digit of the grip fraction moves the planned corner speeds by around a millimetre per second, and the telemetry shows it.
`0.6000000000000001` and `0.6` race to the same 93.95 s, and differ in their rows.
In the record the two give the same race time in all 15 of their 3-lap races (training and validation), with different telemetry every time.
"A different race" here means **different telemetry**, not a different time.
The rule: snap every searched value to its grid (`round(x, 2)`), write it with `repr` (`variant` does), and the number in the file is the number you meant.

**Trap 3: ticks.**
A race time is a whole number of control ticks times 0.05 s, so 5593 ticks prints as `279.65000000000003`.
Compare times with a tolerance (this book pins them to one tick, `TICK`), never with `==` on a computed sum.
And a gain smaller than one tick cannot be seen at all: two settings 0.02 s apart finish on the same tick and tie.

**Trap 4: timing on is not reproducible.**
With `MPC_BUDGET_MS` set, the agent reads the clock: an `mpc()` call slower than the budget halves the horizon (Unit 10).
How slow a call is depends on the machine and its load, so the **same file and the same seed** can race differently.
That is why every race in this unit runs with the timing off.
Here is the record (p6), not raced: a budget of 0.85 ms, near `mpc()`'s own cost, four runs in a row:

In [ ]:
# The probe record of plan roar-012 (p6): agents/unit-11.py with MPC_BUDGET_MS = 0.85, test-circuit-seed1, 1 lap,
# race seed 0, four runs one after another on the same machine. Shown, not raced, and not asserted: it is load-dependent.
TIMING_ON = [  # (race time in s, a fingerprint of the telemetry, the horizon halvings printed during the race)
    (95.25, "730459e3c56c", "40 -> 20 after a 1.10 ms call, then 20 -> 10"),
    (95.25, "8e14d17faf87", "40 -> 20 after a 0.91 ms call"),
    (95.25, "23703647733b", "40 -> 20 after a 0.93 ms call"),
    (95.25, "b16e90da25e2", "40 -> 20 after a 0.87 ms call"),
]
TIMING_OFF = (95.20, "0c2187e67232")
for run, (time_s, fingerprint, halvings) in enumerate(TIMING_ON, start=1):
    print(f"run {run}: {time_s:.2f} s, telemetry {fingerprint}, {halvings}")
print(f"timing off: {TIMING_OFF[0]:.2f} s, telemetry {TIMING_OFF[1]}, every run")
print("different races among the four timing-on runs:", len({fingerprint for _, fingerprint, _ in TIMING_ON}))

Four runs, four different races: the horizon was halved at a different tick each time, once twice.
They happen to share a time; nothing guarantees that.
None of them is the timing-off race.
The shipped file keeps `MPC_BUDGET_MS = 5.0`, well above the slowest `step()` the record measured (1.37 ms), so a race on a normal machine never trips it; but when you tune, switch it off, or your search is scoring the machine's load.

**Trap 5: dead knobs.**
§2: a constant the shipped mode never reads gives the same race whatever its value.
A search over it measures nothing, and a "tuned" value of it is noise you have written into your file.

**Packaging.**
A tuned agent is still one file, and the official runner gives it nothing else.
`agents/unit-12.py` is `agents/unit-11.py` with one constant changed and a header comment that says why:

In [ ]:
eleven, twelve = module_constants(UNIT_11), module_constants(UNIT_12)
changed = {name: (eleven[name], twelve[name]) for name in eleven if eleven[name] != twelve[name]}
print(f"{len(twelve)} module constants; changed from unit-11.py: {changed}")
assert set(eleven) == set(twelve) and list(changed) == ["A_DEC"]
with open(UNIT_12) as handle:
    print("".join(line for _, line in zip(range(6), handle)))

# Write down what the numbers were measured with: a pin for every number in this unit.
python = sys.version_info
print(f"Python {python[0]}.{python[1]}.{python[2]}, numpy {np.__version__}, racekit {racekit.__version__}")

The packaging checklist for a tuned agent:

- **one file**, the official shape (`RoarCompetitionSolution`, `async def initialize` and `step`), importing only the standard library, numpy and `roar_py_interface`, and running on **Python 3.8**;
- **every tuned value on its grid and written with `repr`** (trap 1, trap 2), and a comment saying which search chose it, on what objective, and what it costs;
- **the timing back on as shipped** (`MPC_BUDGET_MS = 5.0`): you tuned with it off, and the race runs with it on;
- **`check_submission` passes** (the milestone runs it on `agents/unit-12.py`);
- **versions pinned**: write down the Python, numpy and racekit versions the numbers were measured with (above).
  Nothing in racekit varies with a version today, and that is exactly why you write it down: so that when a number moves, you can rule the versions out.

| this unit | `unit-11.py` | `unit-12.py` |
|---|---|---|
| the change | (hand-tuned, Units 7 to 11) | `A_DEC` 5.0 → 5.5, by a crash-aware coordinate search |
| crash-aware objective, 16 races (s) | 2224.90 / 2224.35 | 2215.35 / 2215.25 |
| validation, `test-circuit-seed2026` | 282.00 s, 5/5 clean | **280.65 s**, 5/5 clean |
| test, `test-circuit-seed2028` | 271.05 s, 5/5 clean | **269.65 s**, 5/5 clean |
| Unit 11's held-out cars, clean | **24/24** | 20/24 (0 collisions) |

**What this unit adds to the toolbox:**

- race time as a **black-box objective**: per-seed sums over a set of tracks, compared by Unit 1's rule, any race that is not clean rejecting the setting;
- a **knob map**: which constants act, which only after a respawn, which never; "dead" decided by telemetry, not by time;
- **coordinate, random and evolution-strategy search** on a race budget, and a taste of **policy gradient**;
- **overfitting**, and its defence: **training, validation and test tracks**, the test raced once, at the end;
- a **crash-aware objective**, and the honest cost of shipping at its edge;
- **profiling** `step()`: where the tick goes, dead work, and the horizon trade;
- **determinism**: the same seed is the same race, timing off; `repr`, grid snapping and ticks; and a packaging checklist.

Next: **Unit 13, race day**: one track you have never raced, `monza-like`, and every tool in the book.